# Python Memory Model, References, GC and Performance Internals

**Level:** Advanced &nbsp;|&nbsp; **Foldelay:** `Python/Advance` &nbsp;|&nbsp; **Python:** 3.14 (CPython)

this notebook Python of *inside* jhankne of for is. Zyada tar log Python 5 saal likhte are
magar this no jaante of `a = b` ne asal me what did, `copy()` kaafi tha ya `deepcopy()`,
memory where lag rahi is, and code slow why is. Yahan hum wohi khaali jagah bharenge.

## This notebook me what seekhenge

- **Names vs objects vs references** — Python me "variable" naam of thing asal me what is
- **Assignment semantics** — rebinding vs mutation (interview of favourite)
- **Reference counting** — `sys.getrefcount`, immortal objects (PEP 683)
- **Identity** — `id()`, `is` vs `==`, id-reuse of dhoka
- **Small int cache** and **string interning** (`sys.intern`)
- **Mutable vs immutable** of asli wajah (hashing, sharing, safety)
- **function argument passing** — "pass-by-object-reference" of 3 proofs
- **Mutable default argument trap** — deep dive with `__defaults__`
- **Shallow vs deep copy internals** — `copy`, `__copy__`, `__deepcopy__`, recursive structures, memo
- **Garbage collector** — refcount + cycle detector, generational/incremental, `gc` module, `__del__` of khatray
- **weakref** & `WeakValueDictionary` — cache jo memory leak no kare
- **`__slots__`** — memory and speed both of measurement
- **`sys.getsizeof` vs real size** — recursive sizer khud likhenge
- **CPython object overhead**, list over-allocation
- **list vs tuple vs array vs deque** memory comparison
- **String concat in loop vs `join`** — `timeit` from sabit
- **`dis` from bytecode peek**, constant folding / peephole optimizer
- **Local vs global lookup speed** — `LOAD_FAST` vs `LOAD_GLOBAL`
- **LeetCode:** 138 (Copy list with Random Pointer), 133 (Clone Graph), 146 (LRU Cache)

## Prerequisites

Functions, classes, dict/list/tuple, decorators of basic undelaystanding. `timeit` of naam suna be.

> **Note:** Is notebook me some numbers (refcount, memory sizes, timings) tumhare machine but
> thore mukhtalif aa can are — this normal is. Concept see, exact number ratna no.

---
## 1. Names vs Objects vs References

### what is
Python me **variable ek dabba (box) no is** jis me value rakhi be. Python me:

- **object** = heap but asli thing (value + type + refcount)
- **Name** = ek *label / sticky note* jo kisi object but chipka happened is
- **Reference** = us label of object tak pointer

C/Java of mental model: `int x = 5` → memory me `x` naam of dabba, us me 5.
Python of mental model: heap me `5` naam of object, and `x` label us but chipka.

### why important is
Isi ek baat from this all undelaystanding aata is: `a = b` copy no does, list function me
"badal" why jaati hi, `is` and `==` me farq what is, and deep copy when should.

### Syntax
```python
x = 10        # object 10 create (ya cached lo), naam x us but chipkao
y = x         # new object no — sirf ek and label usi object but
```

In [ ]:
# One object, do naam --------------------------------------------------------
a = [10, 20, 30]      # heap but ek list object bana; naam 'a' us but chipka
b = a                 # koi copy no happened — 'b' bhi usi object but chipak gaya

print("a       =", a)
print("b       =", b)
print("id(a)   =", id(a))          # id() = object of unique pehchan (CPython me memory address)
print("id(b)   =", id(b))
print("a is b  =", a is b)         # is  -> same object? -> True
print("a == b  =", a == b)         # == -> same value?  -> True

# ab object then MUTATE do (object of inside tabdeeli)
b.append(40)
print("\nb.append(40) of after:")
print("a =", a, "  <-- 'a' bhi badal gaya, because object aik hi is")
print("b =", b)

# lesson: naam copy are are, objects no

### output of matlab
`id(a) == id(b)` and `a is b == True` of matlab: heap but **ek hi list** is jis but do labels
lage are. Is for `b.append(40)` ne `a` then bhi "badla" — asal me `a` badla hi no,
**object** badla, and both naam usi object then dekh rahe are.

### Gotcha
Beginners kehte are "list function me by-reference jaati is, int by-value". **Ghalat.**
both barabar reference from go are — farq sirf this is of `int` **immutable** is, is for
it badalna mumkin no, sirf naam then naye object but chipkaya ja can is.

### Interview tip
speak: *"Python names are references then heap objects; assignment binds a name, it never copies
the object. Behaviour differences come from mutability, not from a value/reference distinction."*

---
## 2. Assignment Semantics — Rebinding vs Mutation

### what is
Do exactly mukhtalif operations jo dekhne me ek like lagte are:

| Operation | Naam of work | object of work | Doosre naamon but asar |
|---|---|---|---|
| `x = new_value` | naam then **naye object** but chipkao (**rebinding**) | old object like tha waisa | **koi asar no** |
| `x.append(1)` / `x[0]=1` | naam wahi rehta | object **inside from badla** (**mutation**) | **all then dikhta is** |

`x += y` dhoka is: list but this *mutation* (`__iadd__`, in-place extend) is,
int/tuple/str but *rebinding* (new object) is.

In [ ]:
# Rebinding vs Mutation of farq -------------------------------------------
lst1 = [1, 2]
lst2 = lst1
print("shuru me:      lst1", lst1, " lst2", lst2, " same object?", lst1 is lst2)

lst1 = lst1 + [3]            # + new object creates is -> REBINDING
print("lst1 = lst1+[3]:  lst1", lst1, " lst2", lst2, " same object?", lst1 is lst2)

lst3 = [1, 2]
lst4 = lst3
lst3 += [3]                  # += list but __iadd__ = IN-PLACE MUTATION
print("lst3 += [3]:      lst3", lst3, " lst4", lst4, " same object?", lst3 is lst4)

# int but += hamesha rebinding (int immutable is)
n = 5
before = id(n)
n += 1
print("\nint: id badla?", before != id(n), "-> yaani new object bana")

# tuple of inside list: tuple immutable, magar uska element mutable be can is
t = (1, [2, 3])
t[1].append(4)               # this JAIZ is -> list then mutate did, tuple then no
print("tuple of inside list mutate:", t)
try:
    t[0] = 99                # this NAJAIZ -> tuple of slot badalna
except Typeerror as e:
    print("t[0] = 99 ->", type(e).__name__, ":", e)

### output of matlab
`lst1 = lst1 + [3]` of after `lst1 is lst2` **False** be gaya — because `+` ne **new** object
banew and `lst1` naam us naye but chala gaya; `lst2` purane but hi is.
Magar `lst3 += [3]` of after both ab bhi **same object** are and `lst4` me bhi 3 nazar aayega.

### Common mistake
function of inside `param = param + [x]` likh kar log samajhte are caller of list update be gayi.
no happened — wo rebinding thi. `param.append(x)` ya `param += [x]` do then is.

### Interview tip
*"`+` returns a new object, `+=` calls `__iadd__` which mutates in place for mutable types and
falls back then `__add__` + rebinding for immutable types."*

---
## 3. Reference Counting — CPython memory management of reeh of haddi

### what is
every CPython object of sath ek **counter** is is: kitne references us but lage are.
Counter `0` are hi object of memory **foran** free be jaati is (deterministic!).

### why
this "instant cleanup" is — is for Python me file band to do of for GC of infastar no then do parta
(magar phir bhi `with` use do!). Cycles is from hal no are — us of for alag cycle collector is (section 10).

### Syntax
```python
sys.getrefcount(obj)   # NOTE: khud argument bhi ek reference creates is -> jawab me +1
```

### Immortal objects (PEP 683, Python 3.12+)
`None`, `True`, `False`, chhote ints, chhoti strings ab **immortal** are — their refcount
jaan boojh kar ek bohat bara constant (`3221225472` like) keep diya gaya is taake
multi-threaded programs me counter but lock lagane of need no pare. Is for their
refcount dekh kar ghabraana no.

In [ ]:
import sys

# Normal object of refcount ---------------------------------------------------
data = {"k": "v"}
print("1 naam:            ", sys.getrefcount(data), " (+1 because argument bhi reference is)")

alias = data
print("2 naam:            ", sys.getrefcount(data))

holdelay = [data, data]                 # list of inside 2 references
print("list me 2 and:     ", sys.getrefcount(data))

del alias
print("del alias of after: ", sys.getrefcount(data))

holdelay.clear()
print("holdelay.clear():    ", sys.getrefcount(data))

# Immortal objects (PEP 683) --------------------------------------------------
print("\n--- immortal objects ---")
for obj, label in [(None, "None"), (True, "True"), (1, "int 1"), (256, "int 256")]:
    print(f"{label:>8}: {sys.getrefcount(obj)}")
print("this bohat bare/fixed numbers = immortal. In but assert lagana bekaar is.")

# refcount 0 hone but foran cleanup (deterministic destruction)
class Loud:
    def __init__(self, name):
        self.name = name
    def __del__(self):                 # refcount 0 hone but CPython yahan aata is
        print(f"   [__del__] {self.name} of memory free be gayi")

print("\n--- deterministic cleanup ---")
x = Loud("obj-A")
y = x                                  # refcount 2
print("del x  (refcount 2 -> 1): some no hona should")
del x
print("del y  (refcount 1 -> 0): abhi __del__ will run")
del y
print("saw? cleanup exactly wahan happened jahan aakhri reference gaya")

### output of matlab
every new reference (`alias`, list of slots) counter barhaata is, every `del`/scope-exit ghataata is.
`Loud` class ne dikhaya of aakhri reference go hi `__del__` **usi lamhe** chala —
Java/C# of type "kabhi after me" no.

### Gotchas
1. `sys.getrefcount()` of jawab **hamesha +1** is is (function argument bhi reference is).
2. exception traceback, module globals, and interpreter caches bhi chupke from references rakhte are —
   number expected from zyada aa can is.
3. Free-threaded build (PEP 703) me refcounting of mechanism mukhtalif is (biased/deferred).

### Interview tip
*"CPython uses reference counting for immediate reclamation plus a generational cycle collector
for reference cycles. Refcounting is deterministic but can't free cycles; the GC handles those."*

---
## 4. object Identity — `id()`, `is` vs `==`

### what is
- `id(obj)` → object of unique integer (CPython me memory address).
- `is` → **identity** compare (same object?) — `id(a) == id(b)`
- `==` → **value** compare (`__eq__` call is is)

### why / where
`is` sirf **singletons** of sath: `None`, `True`, `False`, sentinel objects.
Numbers/strings of for hamesha `==`.

### all from bara dhoka: id-reuse
object mar jaye then uska address dobara istemal be can is. Is for purane `id()` then
sambhaal kar then keep and after me compare then do **bug** is.

In [ ]:
# is vs == --------------------------------------------------------------------
a = [1, 2, 3]
b = [1, 2, 3]        # alag object, wohi value
c = a

print("a == b:", a == b, "  (value same)")
print("a is b:", a is b, "  (object alag)")
print("a is c:", a is c, "  (same object)")

# None of for hamesha 'is'
val = None
print("\nval is None:", val is None)

# khatarnaak: == then override did ja can is, is then no
class Weird:
    def __eq__(self, other):
        return True                    # all of barabar hone of daawa!
w = Weird()
print("w == 42     :", w == 42, " <- __eq__ jhooth bol raha is")
print("w is 42     :", w is 42, " <- identity jhooth no bol can")

# id-reuse of dhoka
print("\n--- id reuse ---")
old_id = id([1, 2, 3])                 # this list foran mar gayi (koi reference no)
fresh = [9, 9, 9]                      # new object usi jagah ban can is
print("old id ==  naye object of id?", old_id == id(fresh))
print("Sabaq: id() then store kar of after me compare then do galat is (object zinda rakho warna address recycle).")

# sentinel pattern — 'is' of sahi istemal
MISSING = object()                     # unique sentinel
def get(d, key, default=MISSING):
    v = d.get(key, MISSING)
    if v is MISSING:                   # None bhi valid value be can is, is for sentinel
        return None if default is MISSING else default
    return v
print("\nsentinel:", get({"a": None}, "a"), get({}, "b", "fallback"))

### output of matlab
`a == b` True magar `a is b` False → do alag objects, same value. `Weird` ne dikhaya of `==`
class of control me is (jhooth bol can is) magar `is` interpreter of control me.

### Common mistakes
- `if x is 5:` / `if s is "hello":` — kabhi no. Chhote ints/strings but *ittefaqan* chal goes is,
  bare values but fail be goes is (Python 3.8+ me `SyntaxWarning` bhi gets is).
- `id()` then object of "permanent ID" then undelaystand — object marne of after address recycle is is.

### Interview tip
*"`is` compares identity, `==` compares value via `__eq__`. Use `is` only for singletons/sentinels."*

---
## 5. Small Integer Cache and String Interning

### what is
CPython start are hi **-5 from 256** tak of ints first from bana takes is (small int cache).
Isi type identifier jaisi strings (compile-time literals) **interned** is are — ek hi copy.

### why
this values program me hazaaron dafa aati are; ek hi object share then do memory and time bachata is.
Interned strings of comparison **pointer comparison** from shuru is is → dict lookup fast.

### Syntax
```python
sys.intern(s)   # string then interning table me daal do (ya wahan from maujooda lo)
```

In [ ]:
import sys

# Small int cache -------------------------------------------------------------
x, y = 256, 256
print("256 is 256 :", x is y, " <- cached range (-5..256)")
p, q = 257, 257
print("257 is 257 :", p is q, " <- alag statements/objects be can are")
print("range check:", [(n, id(n) == id(n + 0)) for n in (-5, 0, 100, 256)][:2], "...")

# One hi cell/line me constants compile-time but fold be kar share be go are,
# is for 'is' of natija context but depend does is -> is but bharosa no do.

# String interning ------------------------------------------------------------
s1 = "hello_world"          # identifier like literal -> interned
s2 = "hello_world"
print("\nliteral interning:", s1 is s2)

t1 = "hello world!"          # space/!' that string
t2 = "".join(["hello", " ", "world", "!"])   # runtime but bani
print("runtime built  is  literal:", t1 is t2, " (aksar False)")
print("values equal            :", t1 == t2)

i1, i2 = sys.intern(t1), sys.intern(t2)
print("sys.intern of after      :", i1 is i2, " <- ab ek hi object")

# where faida: bohat si repeated string keys (log parsing, CSV columns, JSON keys)
rows = [{"user_id": i, "status": sys.intern("active" if i % 2 else "inactive")} for i in range(5)]
statuses = [r["status"] for r in rows]
print("\ninterned statuses ek hi object share do are:",
      statuses[1] is statuses[3])

# memory of faida: 100k rows me 'status' of 100k copies of jagah 2 objects
big_naive = ["pending_review_state" + "" for _ in range(3)]          # naye objects ban can
big_intern = [sys.intern("pending_review_state") for _ in range(3)]
print("naive all-same-object? ", all(s is big_naive[0] for s in big_naive))
print("intern all-same-object?", all(s is big_intern[0] for s in big_intern))

### output of matlab
`256 is 256` → True (cache), magar runtime but bani strings of `is` aksar False.
`sys.intern()` of after both naam ek hi object but aa gaye.

### Gotchas
- Interning/caching **implementation detail** is — *optimization* of for theek, *logic* of for kabhi no.
- `sys.intern` sirf tab use do jab bohat si duplicate strings be (parsing, ETL). Warna faida no.
- Interned strings kabhi free no is (process of umar tak) — random user input intern no do,
  warna memory barhti rahegi.

### Interview tip
*"Small ints (-5..256) and compile-time string literals are cached/interned by CPython.
`sys.intern` is a memory/lookup optimisation for many duplicate strings — never rely on it for equality."*

---
## 6. Mutable vs Immutable — asli wajah

### what is
| Immutable | Mutable |
|---|---|
| `int`, `float`, `bool`, `str`, `bytes`, `tuple`, `frozenset`, `complex`, `range` | `list`, `dict`, `set`, `bytearray`, most custom classes |

### Asli wajah (sirf "badal no can" no)
1. **Hashability** — dict key / set member banne of for value of hash zindagi bhar same honi should.
   Mutable object of value badal jati is → hash badal jati → dict me kho goes. Is for
   mutable types `__hash__ = None` rakhte are.
2. **Safe sharing** — immutable object then without dar share/cache/intern did ja can is (thread-safe read).
3. **Optimizations** — constant folding, interning, `tuple` reuse — all immutability but mabni.

In [ ]:
# Hashability of asli imtihan -------------------------------------------------
print("hash('abc') :", hash("abc") is not None)
print("hash((1,2)) :", hash((1, 2)))

try:
    hash([1, 2])                       # list mutable -> hash no
except Typeerror as e:
    print("hash([1,2]) ->", type(e).__name__, ":", e)

print("\nlist.__hash__ :", list.__hash__, " <- jaan boojh kar None")
print("tuple.__hash__:", tuple.__hash__ is not None)

# tuple hashable is... but sirf tab jab uske inside all hashable be
try:
    hash((1, [2]))
except Typeerror as e:
    print("hash((1,[2])) ->", type(e).__name__, ":", e)

# if mutable object then key banane of ijazat is then what is (simulate)
class BadKey:
    def __init__(self, v): self.v = v
    def __hash__(self): return hash(self.v)      # hash mutable state but -> KHATRA
    def __eq__(self, other): return isinstance(other, BadKey) and self.v == other.v

k = BadKey(1)
d = {k: "got"}
print("\nlookup first :", d.get(k))
k.v = 99                                # state badal di -> hash badal gaya
print("state badalne of after lookup:", d.get(k), " <- object kho gaya!")
print("magar wo ab bhi dict me maujood is:", list(d.values()))
print("Yehi wajah is of list/dict/set hashable no are.")

# Immutable of matlab 'deeply frozen' no is
t = (1, [2, 3])
t[1].append(4)
print("\ntuple immutable is magar deep-immutable no:", t)

### output of matlab
`BadKey` demo Python of core reason dikhata is: hash badal jaye then object apne bucket me
"kho goes" is — dict me is magar `get()` no milta. Is for Python mutable containers then
hashable hone hi no gives.

### Interview tip
*"Immutability isn't about being 'read-only' for convenience — it's what makes an object safely
hashable, shareable and cacheable. `tuple` is immutable but not deeply immutable."*

---
## 7. function argument Passing — "pass by object Reference"

### what is
Python no pure "pass by value" is no "pass by reference". Sahi naam:
**call by object reference** (ya "call by sharing").

Jab tum `f(x)` likhte be:
- function of parameter ek **new local naam** is
- wo naam **usi object** but chipakta is jo caller of paas tha

Nateeja:
- object then **mutate** do → caller then dikhega
- parameter then **rebind** do (`param = ...`) → caller then some no dikhega

In [ ]:
# Proof 1: rebinding caller then nazar no aati -------------------------------
def rebind(items):
    print("   inside (shuru) id:", id(items))
    items = [99]                    # naye object but chipak gaya -> local baat
    print("   inside (after)  id:", id(items))
    return items

outer = [1, 2, 3]
print("outside id       :", id(outer))
rebind(outer)
print("call of after   :", outer, " <- exactly no badla\n")

# Proof 2: mutation caller then nazar aati is
def mutate(items):
    items.append(99)                # object of inside tabdeeli
print("mutate from first:", outer)
mutate(outer)
print("mutate of after :", outer, " <- badal gaya\n")

# Proof 3: id() from sabit of object share be raha is
def show_id(obj):
    return id(obj)
d = {"a": 1}
print("same object hi gaya?", show_id(d) == id(d))

# Immutable of sath: mutation mumkin hi no, is for 'all some safe' lagta is
def try_change_number(n):
    n += 1                          # rebinding (int immutable)
    return n
num = 10
print("\nint:", try_change_number(num), "| outside that:", num)

# Sahi method jab caller of list then chhoona no be: new value return do
def add_item_pure(items, x):
    return items + [x]              # new object -> caller of list mehfooz
base = [1, 2]
new = add_item_pure(base, 3)
print("pure style -> base:", base, " new:", new)

### output of matlab
Teen proofs: (1) `items = [99]` of outside zero asar, (2) `items.append(99)` of outside poora asar,
(3) `id()` barabar → object copy no happened, share happened.

### Gotchas
- argument then mutate then do (side effect) theek is magar **document do**. Library code me
  in-place functions then `sort_in_place` / `update_*` like clean naam do.
- Defensive copy: `def f(items): items = list(items)` — jab caller of list then chhoona no chahte.

### Interview tip
*"Arguments are passed by object reference: the parameter is a new local name bound then the same
object. Mutations are visible then the caller; rebinding is not."*

---
## 8. Mutable Default argument Trap — deep dive

### what is
```python
def f(items=[]):   # <-- bug
```
Default value **function define are time EK dafa** evaluate is is and function object of
`__defaults__` tuple me sambhaal li jaati is. every call me **wohi** list dobara use is is.

### why aisa is (design wajah)
`def` ek executable statement is; defaults usi time evaluate are are. every call but
re-evaluate then do semantics then closure like pechida bana gives and slow bhi is.

### Sahi method
`None` sentinel + inside new object create.

In [ ]:
# Trap live -------------------------------------------------------------------
def add_bad(item, bucket=[]):          # bug: mutable default
    bucket.append(item)
    return bucket

print("call 1:", add_bad("a"))
print("call 2:", add_bad("b"), " <- purani list wapas aa gayi!")
print("call 3:", add_bad("c"))
print("function of chhupa happened state:", add_bad.__defaults__)   # yahan zinda baitha is

# Sahi method ---------------------------------------------------------------
def add_good(item, bucket=None):
    if bucket is None:                 # 'is None' -> falsy check from behtar ([] bhi falsy is)
        bucket = []
    bucket.append(item)
    return bucket

print("\ngood call 1:", add_good("a"))
print("good call 2:", add_good("b"), " <- every dafa fresh")
print("caller of list of sath:", add_good("z", ["x", "y"]))

# why 'if not bucket' galat is: caller of khaali list chup-chaap replace be jaati is
def add_subtle_bug(item, bucket=None):
    if not bucket:                     # [] bhi falsy -> caller of list ignore!
        bucket = []
    bucket.append(item)
    return bucket

callers_list = []
add_subtle_bug("a", callers_list)
print("\n'if not bucket' of sath caller of list:", callers_list, " <- update no happened (bug)")

# this masla sirf list/dict of no -- every us expression of is jise fresh should
import time
def stamped(t=time.time()):            # bug: define-time of time hamesha
    return t
first = stamped(); time.sleep(0.01); second = stamped()
print("\ntime default same?", first == second, " <- 'now' no milta, define-time milta is")

# Immutable default exactly safe is
def safe(n, factor=2, label="row"):    # int/str immutable -> koi masla no
    return f"{label}-{n * factor}"
print(safe(3), safe(3, 3))

# class body me bhi wahi trap (shared class attribute)
class Basket:
    items = []                         # bug: all instances ek hi list share will do
    def add(self, x): self.items.append(x)
b1, b2 = Basket(), Basket()
b1.add("apple")
print("\nclass attribute trap -> b2.items:", b2.items, " <- b2 ne some add hi no did tha")

### output of matlab
`add_bad.__defaults__` me wohi list nazar aa rahi is jo call-then-call bharti ja rahi is —
yaani "chhupa happened global state". `Basket` demo ne dikhaya of class body me bhi yehi trap is.

### Gotchas
1. `if not bucket:` vs `if bucket is None:` — pehla caller of *khaali* list discard kar gives is.
2. `def f(x, _cache={})` jaan boojh kar cache of for runs is, magar `functools.cache` clean is.
3. Mutable class attributes of for `__init__` me `self.items = []` write.

### Interview tip
*"Default arguments are evaluated once at definition time and stored in `__defaults__`.
Use `None` as a sentinel for mutable defaults."*

---
## 9. Shallow vs Deep Copy — internals

### what is
- **Shallow copy** — new *container*, inside those objects wohi (shared).
  `list(x)`, `x[:]`, `dict(x)`, `x.copy()`, `copy.copy(x)`
- **Deep copy** — new container **and** inside all some recursively new. `copy.deepcopy(x)`

### deepcopy of internal jadoo: `memo` dict
`deepcopy` ek `memo` (id → nayi copy) dictionary holds is:
1. **Cycles** infinite recursion me no go
2. Ek hi object if do jagah tha, then copy me bhi **ek hi** (shared) rahega — structure mehfooz

### when kaunsa
- Flat data (numbers/strings of list) → shallow kaafi and bohat fast
- Nested config/tree/graph jise tum modifygi → deep
- `deepcopy` **slow** is; hot loop me no run

In [ ]:
import copy

original = {"name": "ali", "scores": [10, 20], "meta": {"city": "Lahore"}}

shallow = copy.copy(original)          # == dict(original) == original.copy()
deep = copy.deepcopy(original)

print("shallow is original:", shallow is original)
print("deep    is original:", deep is original)

# inside those objects?
print("\nnested list shared? shallow:", shallow["scores"] is original["scores"])
print("nested list shared? deep   :", deep["scores"] is original["scores"])

# ab mutate kar of asar see
original["scores"].append(30)
original["meta"]["city"] = "Karachi"
print("\noriginal:", original)
print("shallow :", shallow, " <- with with badal gaya")
print("deep    :", deep, " <- exactly mehfooz")

# top-level rebinding shallow copy then no chhoota
original["name"] = "usman"
print("\nname rebind of after -> shallow['name']:", shallow["name"])

In [ ]:
import copy

# Structure sharing of proof: memo dict of kamaal ----------------------------
inner = [1, 2]
nested = [inner, inner]                # ek hi list, do jagah
d = copy.deepcopy(nested)
print("original me both slots same object?:", nested[0] is nested[1])
print("copy me bhi both slots same object?:", d[0] is d[1], " <- memo ne structure bachaya")
print("copy original from alag is?         :", d[0] is not inner)

# Recursive / self-referencing structure --------------------------------------
loop = [1, 2]
loop.append(loop)                      # khud then contain does is!
print("\nself-referencing list:", str(loop)[:30], "...")
lc = copy.deepcopy(loop)               # crash no will be -> memo of wajah from
print("deepcopy of after khud then contain does is?", lc[2] is lc)
print("original from alag?", lc is not loop)

# list of shallow copy of 4 tareeqe
src = [1, 2, 3]
copies = [src[:], list(src), src.copy(), copy.copy(src)]
print("\nsab equal:", all(c == src for c in copies),
      "| all alag object:", all(c is not src for c in copies))

# 2D grid of classic bug
grid_bug = [[0] * 3] * 3               # ek hi row 3 dafa!
grid_bug[0][0] = 9
print("\n[[0]*3]*3 of sath:", grid_bug, " <- poora column badal gaya")
grid_ok = [[0] * 3 for _ in range(3)]  # every row alag object
grid_ok[0][0] = 9
print("comprehension of sath:", grid_ok)

### output of matlab
- Shallow copy me `scores` list **share** thi → original of `append` copy me bhi dikha.
- Deep copy poori type alag thi.
- `memo` of wajah from (a) self-referencing list `Recursionerror` of baghair copy happened,
  (b) "do slots me ek hi object" that rishta copy me bhi qaayam raha.

### Gotchas
- `copy.copy` custom class but default me `__dict__` shallow copy does is — nested attributes share honge.
- `deepcopy` un objects but masla does is jo copy hone layak no (open file, socket, lock, DB connection).
- `deepcopy` **slow** is (recursion + memo). Config load but theek, per-request loop me no.

### Interview tip
*"`deepcopy` uses a memo dict keyed by `id()`, which both prevents infinite recursion on cycles
and preserves shared-object structure in the copy."*

---
## 10. `__copy__` and `__deepcopy__` hooks — copy then control then do

### what is
`copy.copy(obj)` first `obj.__copy__()` dhoondta is; `copy.deepcopy(obj)`
`obj.__deepcopy__(memo)` dhoondta is. no milne but default `__reduce_ex__`/`__dict__` that rasta.

### where need parti is
- object me **non-copyable resource** be (file handle, socket, lock, connection pool)
- Copy me some fields **share** rakhni be and some fresh
- Copy do time cache/ID **reset** then do be

In [ ]:
import copy, threading

class Report:
    # One class jis me non-copyable resource (lock) and ek cache is
    def __init__(self, title, rows):
        self.title = title
        self.rows = rows
        self._lock = threading.Lock()      # Lock deepcopy-friendly no is
        self._cache = {"total": sum(rows)} # copy me reset then do behtar

    def __copy__(self):
        cls = self.__class__
        new = cls.__new__(cls)             # __init__ bypass
        new.__dict__.update(self.__dict__) # all some shallow (rows SHARED)
        new._lock = threading.Lock()       # resource fresh
        return new

    def __deepcopy__(self, memo):
        cls = self.__class__
        new = cls.__new__(cls)
        memo[id(self)] = new               # LAZMI first: cycles of for
        new.title = copy.deepcopy(self.title, memo)
        new.rows = copy.deepcopy(self.rows, memo)
        new._lock = threading.Lock()       # new lock, copy no
        new._cache = {}                    # cache reset
        return new

    def __repr__(self):
        return f"Report({self.title!r}, rows={self.rows}, cache={self._cache})"

r = Report("Q1", [1, 2, 3])
sc = copy.copy(r)
dc = copy.deepcopy(r)

print("original :", r)
print("shallow  :", sc, "| rows shared?", sc.rows is r.rows, "| lock same?", sc._lock is r._lock)
print("deep     :", dc, "| rows shared?", dc.rows is r.rows, "| lock same?", dc._lock is r._lock)

r.rows.append(4)
print("\noriginal me append of after -> shallow.rows:", sc.rows, " deep.rows:", dc.rows)

# without hook, lock that object deepcopy but error gives is
class Naive:
    def __init__(self):
        self.lock = threading.Lock()
try:
    copy.deepcopy(Naive())
    print("\n(is build me lock copy be gaya -- phir bhi rely no do)")
except exception as e:
    print("\nwithout hook deepcopy(lock) ->", type(e).__name__, ":", str(e)[:70])

### output of matlab
`__copy__` ne rows share of magar lock new diya; `__deepcopy__` ne all new diya and cache reset did.

### Gotcha
`__deepcopy__` me **all from first** `memo[id(self)] = new` write, warna cyclic structure
infinite recursion me chala jayega.

### Interview tip
*"Override `__copy__`/`__deepcopy__` when your object owns resources that shouldn't be duplicated;
register yourself in `memo` first then stay cycle-safe."*

---
## 11. Garbage Collector — refcount + cycle detector

### what is
CPython me **do** cleanup mechanisms are:

1. **Reference counting** (primary) — refcount 0 → foran free. Fast, deterministic.
2. **Cycle collector** (`gc` module) — un objects of for jo aik doosre then pakar kar baithe are
   (`a.peer = b; b.peer = a`). In of refcount kabhi 0 no is, is for alag detector should.

### Generational idea
Objects "generations" me rakhe go are (0 = naye). Zyada tar objects jawan hi mar go are
(*weak generational hypothesis*), is for gen-0 baar baar and purani generations kam kam scan is are.
**Python 3.13+ me GC incremental is** — is for `gc.get_threshold()` of teesri value `0` dikhti is;
purani "700, 10, 10" that baat ratna no, khud print kar of see.

### Ahem
Sirf **container** objects (jo doosre objects then refer kar can are) GC track does is.
`int`/`str` like atomic objects cycle bana hi no can → track no are.

In [ ]:
import gc, weakref

print("gc enabled?      :", gc.isenabled())
print("thresholds       :", gc.get_threshold(), " (3.13+ incremental GC)")
print("gen counts       :", gc.get_count())
print("tracked? list    :", gc.is_tracked([1, 2]))
print("tracked? int     :", gc.is_tracked(42), " <- atomic objects track no are")

# ---- Reference cycle live demo ----
class Node:
    def __init__(self, name):
        self.name = name
        self.peer = None
    def __repr__(self):
        return f"Node({self.name})"

a, b = Node("A"), Node("B")
a.peer = b
b.peer = a                    # cycle: A -> B -> A

watch_a = weakref.ref(a)      # weakref refcount no barhata -> jasoosi of for perfect
del a, b                      # both naam gaye... magar objects aik doosre then pakre are

alive_before = watch_a() is not None
print("\ndel of after zinda is?      ", alive_before, " <- refcounting nakaam (cycle)")
collected = gc.collect()      # cycle collector run
print("gc.collect() ne free kiye   :", collected, "objects")
print("ab zinda is?               ", watch_a() is not None, " <- cycle collector ne clean did")

# ---- Cycle from bachne of method: weakref parent pointer ----
class Child:
    def __init__(self, parent):
        self._parent = weakref.ref(parent)     # strong ref no -> koi cycle no
    @property
    def parent(self):
        return self._parent()                  # call do -> object ya None

class Parent:
    def __init__(self):
        self.children = []
    def add(self):
        c = Child(self); self.children.append(c); return c

p = Parent(); c = p.add()
watch_p = weakref.ref(p)
print("\nchild from parent milta is?  ", c.parent is p)
del p
print("parent del of after (without gc):", watch_p(), " <- foran free, cycle bana hi no")
print("child of parent ab           :", c.parent)

### output of matlab
`del a, b` of after bhi object zinda tha (aik doosre of wajah from refcount 1 baaqi tha).
`gc.collect()` ne cycle detect kar of free did. Doosre demo me `weakref` of wajah from cycle
bana hi no — object foran free be gaya, GC of need hi no pari.

### Gotchas
- `gc.disable()` sirf tab jab yaqeen be of cycles no banenge (some high-perf short-request services
  do are) — warna memory barhti rahegi.
- Cycles aam are: doubly linked list, parent/child trees, `self.callback = self.method`,
  exception + `__traceback__` (frames then pakarta is), closures.
- Leak hunting: `gc.get_referrers(obj)` / `gc.get_referents(obj)`, and
  `gc.set_debug(gc.DEbug_SAVEALL)` + `gc.garbage`.

### Interview tip
*"Refcounting frees objects immediately but cannot free reference cycles; the generational
(now incremental) `gc` handles cycles and only tracks container objects."*

---
## 12. `__del__` of khatray

### what is
`__del__` "destructor" is — refcount 0 hone but (ya GC of time) runs is.

### why bachna should
1. **time of koi wada no** — interpreter shutdown but exactly no runs then bhi legal is.
2. **Exceptions nigal go are** — sirf "unraisable exception" stdelayr but goes is.
3. **Resurrection** — `__del__` me object then kahin store kar do then wo zinda be goes is.
4. Shutdown but module globals `None` be can are → `__del__` me module use then do crash-prone.

### Sahi rasta
- Resource cleanup → **context manager** (`with`, `__enter__`/`__exit__`) ya explicit `.close()`
- "object mar jaye then this run" → **`weakref.finalize`** (safe, ek dafa, predictable)

In [ ]:
import weakref

# 1) __del__ me exception chup-chaap gum be goes is
class Risky:
    def __del__(self):
        try:
            raise Valueerror("cleanup fail!")   # asli code me this gum be goes
        except exception as e:
            print("   [__del__] exception khud handle did:", e)

r = Risky(); del r
print("saw? program runs raha -- error kahin report no happened\n")

# 2) Resurrection: __del__ object then zinda kar can is
GRAVEYARD = []
class Zombie:
    def __init__(self, n): self.n = n
    def __del__(self):
        GRAVEYARD.append(self)      # KHATRA: khud then zinda kar liya
z = Zombie(1); del z
print("del of after bhi zinda:", len(GRAVEYARD) == 1, "-> and __del__ dobara no will run")
GRAVEYARD.clear()

# 3) Behtar A: weakref.finalize
class Connection:
    def __init__(self, name):
        self.name = name
        # NOTE: callback me self capture no do warna object kabhi marega hi no
        self._fin = weakref.finalize(self, print, f"   [finalize] {name} band be gaya")
    def close(self):
        self._fin()                 # explicit close -> foran, sirf ek dafa

c = Connection("db-1")
print("\nfinalize alive?", c._fin.alive)
c.close()
print("close of after alive?", c._fin.alive)
c2 = Connection("db-2")
del c2                              # yahan finalize khud will run
print()

# 4) Behtar B: context manager (all from pasandeeda)
class Session:
    def __enter__(self):
        print("   [enter] session khula")
        return self
    def __exit__(self, exc_type, exc, tb):
        print("   [exit ] session band (exception be ya no be)")
        return False                # exception dabao dont

with Session() as s:
    print("   work be raha is...")
try:
    with Session():
        raise Runtimeerror("some toot gaya")
except Runtimeerror as e:
    print("   exception outside aaya, magar cleanup first happened:", e)

### output of matlab
`Risky` me error gum be gaya, `Zombie` ne khud then zinda kar liya, jab of `finalize` and `with`
both ne predictable ek-dafa cleanup diya — and `with` ne exception of sath bhi cleanup did.

### Interview tip
*"`__del__` is non-deterministic, swallows exceptions and can resurrect objects. Use context
managers for resource lifetime and `weakref.finalize` for 'run when this dies' hooks."*

---
## 13. `weakref` and `WeakValueDictionary` — memory-safe cache

### what is
**Weak reference** = object of pata jo uska refcount **no** barhata. object mar can is
chahe weakref maujood be; after me weakref call to do but `None` milta is.

### Asli use-cases
1. **Cache / identity map** — jab tak object kahin asal me use be raha is cache me raho,
   warna khud nikal jao (`WeakValueDictionary`)
2. **Parent pointers / back-references** — cycles from bachne of for
3. **Observers / callbacks** — listener then zinda dont rakho
4. **Per-object metadata** — `WeakKeyDictionary` (object → extra info)

### Gotcha
every object weakref-able no: plain `int`/`str`/`tuple`, and `__slots__` that class
(jab tak `__weakref__` slot no be) support no do.

In [ ]:
import weakref, gc

class Record:
    __slots__ = ("rid", "payload", "__weakref__")   # <-- __weakref__ warna weakref fail
    def __init__(self, rid, payload):
        self.rid, self.payload = rid, payload
    def __repr__(self):
        return f"Record({self.rid})"

# 1) Basic weakref
rec = Record(1, "data")
wr = weakref.ref(rec)
print("weakref from object:", wr(), "| alive?", wr() is not None)
del rec
gc.collect()
print("del of after      :", wr(), " <- reference khud dead be gaya\n")

# 2) WeakValueDictionary from identity-map cache
class RecordCache:
    # Maqsad: if same rid of object first from zinda is then wohi do (duplicate memory no lage)
    def __init__(self):
        self._cache = weakref.WeakValueDictionary()
        self.hits = self.misses = 0
    def get(self, rid, payload):
        obj = self._cache.get(rid)
        if obj is not None:
            self.hits += 1
            return obj
        self.misses += 1
        obj = Record(rid, payload)
        self._cache[rid] = obj          # weak -> koi strong ref no bache then nikal jayega
        return obj

cache = RecordCache()
r1 = cache.get(7, "seven")
r2 = cache.get(7, "seven")              # same object milna should
print("both same object?", r1 is r2, "| hits:", cache.hits, "misses:", cache.misses)
print("cache size       :", len(cache._cache))

del r1, r2                              # aakhri strong references gaye
gc.collect()
print("references jaane of after cache size:", len(cache._cache), " <- leak no happened")

# Muqabla: normal dict cache object then zinda pakar takes is
strong = {}
r3 = Record(8, "eight")
strong[8] = r3
del r3
gc.collect()
print("normal dict cache size:", len(strong), " <- object phansa happened (potential leak)")

# 3) WeakKeyDictionary from per-object metadata
meta = weakref.WeakKeyDictionary()
r4 = Record(9, "nine")
meta[r4] = {"last_seen": "2026-01-01"}
print("\nmetadata:", meta[r4], "| entries:", len(meta))
del r4
gc.collect()
print("object marne but metadata bhi gaya:", len(meta))

# 4) Non-weakref-able types
try:
    weakref.ref((1, 2))
except Typeerror as e:
    print("\nweakref.ref((1,2)) ->", type(e).__name__, ":", e)

### output of matlab
`WeakValueDictionary` ne duplicate objects bachaye (`r1 is r2`) **and** references jaane but
khud clean bhi be gayi. Normal `dict` me entry phansi rah gayi — yehi classic
"cache = memory leak" bug is.

### Gotchas
- Weak dict but iterate do time entries beech me gaayab be can are → `list(d.items())` le kar chalo.
- Bounded (LRU) cache should then weakref kaafi no — `functools.lru_cache` ya manual LRU
  (see LC 146 neeche).
- `__slots__` of sath `__weakref__` slot add then do necessary is.

### Interview tip
*"Use `WeakValueDictionary` for identity-map caches: entries vanish when the last strong reference
dies, so the cache can't keep objects alive forever."*

---
## 14. `__slots__` — memory and speed both

### what is
Default me every instance of paas ek **`__dict__`** is is (attributes of dictionary) —
flexible magar mehnga. `__slots__` batata is of attributes of list fixed is, then CPython
`__dict__` of bajaye **fixed-size C array** use does is.

### Faide
1. **Memory** — per-instance dict (~100+ bytes) khatam
2. **Speed** — attribute access thora fast (offset lookup, dict hashing no)
3. **Typo safety** — `obj.nmae = 1` but `Attributeerror` (new attribute ban hi no can)

### Nuqsanat
- Naye attributes add no kar can (dynamic patching band)
- `__weakref__` alag from maangna parta is
- Multiple inheritance me slots conflict pechida
- class-level defaults + slots of naam clash no be can

In [ ]:
import sys, timeit

class PointDict:                        # normal class -> __dict__ of sath
    def __init__(self, x, y):
        self.x, self.y = x, y

class PointSlots:                       # slots that class
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

pd, ps = PointDict(1, 2), PointSlots(1, 2)

# --- MEMORY ---
dict_total = sys.getsizeof(pd) + sys.getsizeof(pd.__dict__)
slots_total = sys.getsizeof(ps)
print("PointDict : object", sys.getsizeof(pd), "+ __dict__", sys.getsizeof(pd.__dict__),
      "=", dict_total, "bytes")
print("PointSlots:", slots_total, "bytes")
print(f"bachat    : {dict_total - slots_total} bytes per instance "
      f"({100 * (1 - slots_total / dict_total):.0f}% kam)")
print("100k objects but bachat: ~", round((dict_total - slots_total) * 100_000 / 1024 / 1024, 1), "MB")

# --- SPEED (sirf print, koi timing assert no) ---
t_dict = timeit.timeit("pd.x", globals=globals(), number=200_000)
t_slot = timeit.timeit("ps.x", globals=globals(), number=200_000)
print(f"\nattribute read: dict {t_dict:.4f}s | slots {t_slot:.4f}s "
      f"(slots aam tor but ~10-25% fast)")

# --- Typo safety ---
try:
    ps.z = 5
except Attributeerror as e:
    print("\nslots but new attribute:", type(e).__name__, ":", e)
pd.z = 5
print("dict that class but chal gaya (typo pakra no gaya):", pd.z)

# --- __dict__ is ya no ---
print("\nhasattr(ps, '__dict__'):", hasattr(ps, "__dict__"))
print("PointSlots.__slots__   :", PointSlots.__slots__)

# --- Inheritance of gotcha: subclass without __slots__ of then dict wapas aa goes is ---
class Child(PointSlots):
    pass                                # __slots__ no diya -> __dict__ wapas
ch = Child(1, 2); ch.anything = 1
print("subclass without __slots__ -> __dict__ wapas:", hasattr(ch, "__dict__"))

class ChildOK(PointSlots):
    __slots__ = ()                      # sahi method: khaali slots
print("subclass with __slots__=() -> __dict__?", hasattr(ChildOK(1, 2), "__dict__"))

### output of matlab
slots those object of size sirf object is (koi alag dict no) — is for 100k rows but
megabytes of bachat. attribute read bhi thora fast. and `ps.z = 5` ne typo pakar liya.

### Gotchas
1. Subclass me `__slots__ = ()` no write then `__dict__` wapas aa goes is and poora faida khatam.
2. `@dataclass(slots=True)` (Python 3.10+) from dataclass of sath slots asaani from get go are.
3. slots + `weakref` should then `__slots__` me `"__weakref__"` add do.
4. slots premature optimization ban can are — sirf tab jab **hazaaron/lakhon** instances be.

### Interview tip
*"`__slots__` replaces the per-instance `__dict__` with a fixed C-level array: less memory,
slightly faster attribute access, no dynamic attributes."*

---
## 15. `sys.getsizeof` vs asli size — recursive sizer

### what is
`sys.getsizeof(obj)` sirf **us object of apna** size gives is — inside those objects of no.
list of for this sirf pointer array of size is, elements of no!

### why matter does is
Memory audit do time log `getsizeof(big_list)` dekh kar khush be go are of "sirf 8KB".
Asal me inside 10,000 strings be can are = megabytes.

### Sahi method
Recursive sizer write jo `id()` of sath **seen** set rakhe (shared objects do dafa no ginay,
cycles me no phanse). Ya `tracemalloc` / `pympler` jaisi thing use do.

In [ ]:
import sys
from itertools import chain
from collections import deque

def deep_size(obj, seen=None):
    # object of 'asli' size: khud + inside those all (shared objects sirf ek dafa)
    seen = set() if seen is None else seen
    oid = id(obj)
    if oid in seen:                      # first gin liya (ya cycle) -> 0
        return 0
    seen.add(oid)
    size = sys.getsizeof(obj)
    if isinstance(obj, dict):
        size += sum(deep_size(k, seen) + deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset, deque)):
        size += sum(deep_size(i, seen) for i in obj)
    elif hasattr(obj, "__dict__"):       # custom object -> uska dict bhi
        size += deep_size(vars(obj), seen)
    elif hasattr(obj, "__slots__"):      # slots those object of fields
        size += sum(deep_size(getattr(obj, s), seen)
                    for s in obj.__slots__ if hasattr(obj, s))
    return size

# Demo: shallow vs deep --------------------------------------------------------
words = [f"user_number_{i}" for i in range(500)]
print("getsizeof(list) :", sys.getsizeof(words), "bytes  <- sirf pointer array")
print("deep_size(list) :", deep_size(words), "bytes  <- strings bhi shaamil")
print("farq            :", deep_size(words) - sys.getsizeof(words), "bytes chhup rahe the")

nested = {"users": [{"id": i, "name": f"name{i}"} for i in range(50)]}
print("\ngetsizeof(dict):", sys.getsizeof(nested), "| deep_size:", deep_size(nested))

# Shared object then do dafa no ginna
shared = ["x" * 1000]
pair = [shared, shared]
print("\nshared list do jagah -> deep_size:", deep_size(pair),
      "(1000-char string sirf ek dafa gini)")

# Cycle safe?
cyc = [1, 2]; cyc.append(cyc)
print("cyclic structure but deep_size chal gaya:", deep_size(cyc), "bytes")

### output of matlab
`getsizeof` ne 500 strings of list then chand hazaar bytes explained, `deep_size` ne asli
(kai guna bara) number diya. `seen` set of wajah from shared string ek dafa gini gayi and
self-referencing list but bhi crash no happened.

### Gotchas
- `getsizeof` custom object of `__dict__` then count no does.
- Interned/shared objects of wajah from "asli" size of tareef hi mushkil is — 2 lists of
  deep_size jama then do double-count kar can is.
- Production memory analysis of for `tracemalloc` (allocations by line) behtar is.

### Interview tip
*"`sys.getsizeof` is shallow — for containers it excludes referenced objects. Deep accounting
needs recursion with an id-based `seen` set, or `tracemalloc`."*

---
## 16. CPython object Overhead and list over-allocation

### what is
Python me "chhoti" thing bhi sasti no: every object of sath **headelay** is is
(refcount + type pointer, and variable-size objects me length).

Approx (64-bit CPython): `object` headelay 16 bytes, `int` 28 bytes (chhota), `float` 24,
khaali `list` 56, khaali `dict` 64, khaali `set` 216, `str` ~41 + chars.

### list over-allocation
`list.append` every dafa exact 1 slot no barhaati — **thora zyada** allocate karti is
(growth pattern ~1.125x) taake every append but realloc no be. Isi for `append` amortized O(1) is.

In [ ]:
import sys

print("--- khaali containers of size ---")
for label, obj in [("int 0", 0), ("int 2**70", 2 ** 70), ("float", 1.5), ("bool", True),
                   ("str ''", ""), ("str 'a'", "a"), ("str 'a'*100", "a" * 100),
                   ("list []", []), ("tuple ()", ()), ("dict {}", {}), ("set()", set()),
                   ("bytes b''", b"")]:
    print(f"{label:>14}: {sys.getsizeof(obj):>4} bytes")

print("\n--- int arbitrary precision: digits of sath size barhta is ---")
for n in (1, 2 ** 30, 2 ** 60, 2 ** 300):
    print(f"  {str(n)[:12]:>14}... -> {sys.getsizeof(n)} bytes")

print("\n--- list over-allocation (append but capacity jumps) ---")
lst = []
last = sys.getsizeof(lst)
print(f"  len=0  size={last}")
for i in range(1, 40):
    lst.append(i)
    cur = sys.getsizeof(lst)
    if cur != last:                    # sirf tab print jab capacity badli
        print(f"  len={i:<3} size={cur}   <- realloc happened (jump {cur - last} bytes)")
        last = cur

print("\n--- exact-size banane of tareeqe ---")
from_list = list(range(20))
from_tuple = tuple(range(20))
print("list(range(20))  :", sys.getsizeof(from_list), "bytes (thori extra capacity)")
print("tuple(range(20)) :", sys.getsizeof(from_tuple), "bytes (exact, immutable)")
print("bachat           :", sys.getsizeof(from_list) - sys.getsizeof(from_tuple), "bytes")

### output of matlab
Capacity jumps sabit do are of `list` "batch" me barhti is. Isi for 1 million appends
but bhi performance theek rehti is (amortized O(1)) — magar memory need from thori zyada.

### Gotchas
- if size first from maloom be, list comprehension / `list(iterable)` faster + tighter is.
- `int` arbitrary precision is — bara number bara object.
- `set` of khaali size all from bara (hash table first from allocate).

### Interview tip
*"CPython lists over-allocate on growth (amortised O(1) append) and store pointers, not values —
which is why a list of ints costs far more than a C array of ints."*

---
## 17. list vs tuple vs array vs deque vs bytes — memory of muqabla

### what is
all "sequence" are magar memory model exactly mukhtalif:

| type | inside what holds is | Mutable? | Best for |
|---|---|---|---|
| `list` | **pointers** (8 bytes) + over-allocation | yes | general purpose, mixed types |
| `tuple` | pointers, exact size | no | fixed records, dict keys |
| `array.array` | **raw C values** (e.g. 4-byte int) | yes | bohat from numbers, homogeneous |
| `bytes` / `bytearray` | raw bytes (1 byte each) | no / yes | withoutry data, buffers |
| `collections.deque` | doubly linked blocks | yes | both taraf from O(1) push/pop |
| `numpy.ndarray` | contiguous C array + vectorized ops | yes | numeric compute |

**Ahem:** `list` me 1000 ints = 1000 pointers **+** 1000 int objects (every ek ~28 bytes).
`array('i')` me wohi 1000 ints = 4000 bytes. Farq 10x from zyada.

In [ ]:
import sys, array
from collections import deque
import numpy as np

N = 1000
as_list  = list(range(N))
as_tuple = tuple(range(N))
as_array = array.array("i", range(N))      # 'i' = signed int (4 bytes)
as_deque = deque(range(N))
as_bytes = bytes(N)                        # N zero bytes
as_np    = np.arange(N, dtype=np.int32)

def total_list_cost(lst):
    # list of apna size + inside those int objects (chhote ints cached are, phir bhi ginte are)
    return sys.getsizeof(lst) + sum(sys.getsizeof(i) for i in lst)

rows = [
    ("list (container only)", sys.getsizeof(as_list)),
    ("list + int objects   ", total_list_cost(as_list)),
    ("tuple (container)    ", sys.getsizeof(as_tuple)),
    ("array('i')           ", sys.getsizeof(as_array)),
    ("deque                ", sys.getsizeof(as_deque)),
    ("bytes                ", sys.getsizeof(as_bytes)),
    ("numpy int32          ", as_np.nbytes + sys.getsizeof(as_np) - as_np.nbytes),
]
print(f"{N} integers rakhne of kharcha:")
for label, size in rows:
    print(f"  {label}: {size:>8} bytes")

print("\nlist+ints vs array ratio:",
      round(total_list_cost(as_list) / sys.getsizeof(as_array), 1), "x")

# deque of asli faida: both sire but O(1)
import timeit
setup_l = "from collections import deque\nl=list(range(10000))\nd=deque(range(10000))"
t_list_front  = timeit.timeit("l.insert(0, 1); l.pop(0)", setup=setup_l, number=2000)
t_deque_front = timeit.timeit("d.appendleft(1); d.popleft()", setup=setup_l, number=2000)
print(f"\nfront insert/pop 2000x -> list {t_list_front:.4f}s | deque {t_deque_front:.4f}s")
print("list.insert(0) O(n) is (all elements shift), deque.appendleft O(1)")

# array/bytes of limitation: sirf ek type
try:
    array.array("i", [1, "do", 3])
except Typeerror as e:
    print("\narray me mixed types ->", type(e).__name__, ":", e)

### output of matlab
`list + int objects` of number `array('i')` from kai guna bara is — because list pointers holds is
and every int alag heap object is. Front insert/pop me `deque` list from bohat fast nikla.

### Gotchas
- Chhote ints cached are are, is for `list(range(1000))` of asli memory shared is is —
  magar bare/unique numbers of sath poora kharcha lagta is.
- `array` sirf numeric homogeneous data of for; math then do be then seedha `numpy` behtar.
- `deque` random indexing (`d[500]`) me list from slow is (O(n)) — trade-off understand.

### Interview tip
*"Python lists store pointers then boxed objects; `array`/`numpy` store raw values contiguously,
which is why they use far less memory and are cache-friendly."*

---
## 18. String concat in loop vs `join` — classic performance trap

### what is
`str` **immutable** is. loop me `s += piece` every dafa **nayi string** creates is and poora
old content copy does is → O(n²) total work.
`"".join(list_of_pieces)` ek dafa total length nikalta is, ek buffer allocate does is → O(n).

### Note
CPython me `s += x` of for ek chhota optimization is (jab refcount 1 be then in-place resize
of koshish) — is for chhote loops me farq chhupa reh can is. Bare data but `join` hi jeetta is,
and `join` of intent bhi clean is.

In [ ]:
import timeit, io

PIECES = [f"line-{i}\n" for i in range(2000)]

def by_concat():
    s = ""
    for p in PIECES:
        s += p                      # every dafa (potentially) nayi string
    return s

def by_join():
    return "".join(PIECES)          # ek allocation

def by_list_append():
    out = []
    for p in PIECES:
        out.append(p)               # list append sasta
    return "".join(out)

def by_stringio():
    buf = io.StringIO()             # file-like buffer (streaming of for behtareen)
    for p in PIECES:
        buf.write(p)
    return buf.getvalue()

# all of natija exactly same hona should
assert by_concat() == by_join() == by_list_append() == by_stringio()
print("chaaron of output same is:", len(by_join()), "chars")

for name, fn in [("concat  +=", by_concat), ("join      ", by_join),
                 ("list+join ", by_list_append), ("StringIO  ", by_stringio)]:
    t = timeit.timeit(fn, number=200)
    print(f"  {name}: {t:.4f}s")

print("\nf-string chhoti cheezon of for all from clean and fast is:")
name, n = "ali", 3
t_fstr = timeit.timeit('f"{name} of {n}"', globals=globals(), number=200_000)
t_plus = timeit.timeit('name + " of " + str(n)', globals=globals(), number=200_000)
t_fmt  = timeit.timeit('"{} of {}".format(name, n)', globals=globals(), number=200_000)
t_pct  = timeit.timeit('"%s of %s" % (name, n)', globals=globals(), number=200_000)
print(f"  f-string {t_fstr:.4f}s | + {t_plus:.4f}s | .format {t_fmt:.4f}s | % {t_pct:.4f}s")

### output of matlab
Chaaron of output barabar is (assert pass happened) magar approach of kharcha mukhtalif.
Bare inputs (100k+ pieces) but `+=` that rasta dramatically slow be goes is.

### Gotchas
- `+=` of CPython optimization sirf tab lagta is jab string of refcount 1 be —
  if tum wo string kahin and bhi keep do (list me append kar do) then optimization khatam.
- Logging me `logger.info("x " + str(y))` no write — `logger.info("x %s", y)` write
  (string tab banti is jab log level allow kare).
- `StringIO` streaming/incremental output of for best (memory me poori list no rakhni parti).

### Interview tip
*"Strings are immutable, so repeated concatenation is O(n²) copying; accumulate pieces in a list
and `join` once, or stream into `io.StringIO`."*

---
## 19. `dis` — bytecode but nazar

### what is
Python source then **bytecode** me compile does is jo CPython of virtual machine chalata is.
`dis` module wohi bytecode dikhata is. this "why this code fast/slow is" of **asli jawab** gives is.

### why seekho
- Pata runs is `LOAD_FAST` (local) vs `LOAD_GLOBAL` of farq
- Constant folding nazar aata is (compiler ne khud hisaab kar liya)
- Comprehension vs loop of difference undelaystanding me aata is

> **Note:** Python 3.11+ me bytecode names badle are (`LOAD_SMALL_INT`, `LOAD_FAST_BORROW`,
> specialized adaptive instructions). Exact opcode names version-specific are — concept see.

In [ ]:
import dis

def add_tax(price):
    tax = price * 0.17          # local variable
    return price + tax

print("=== add_tax of bytecode ===")
dis.dis(add_tax)

print("\n=== code object of inside what is ===")
c = add_tax.__code__
print("co_varnames (locals):", c.co_varnames)
print("co_consts           :", c.co_consts)
print("co_names (globals)  :", c.co_names)
print("stack size          :", c.co_stacksize)

In [ ]:
import dis

# --- Constant folding: compiler khud hisaab kar takes is ---
def seconds_per_day():
    return 24 * 60 * 60         # this runtime but multiply no will be

print("co_consts:", seconds_per_day.__code__.co_consts,
      " <- 86400 first from maujood (constant folding)")
dis.dis(seconds_per_day)

print("\n--- tuple constant bhi fold is is ---")
def const_tuple():
    return (1, 2, 3)
print("co_consts:", const_tuple.__code__.co_consts)

def nonconst_tuple(x):
    return (1, x, 3)            # x runtime but -> tuple runtime but banega
print("variable of sath co_consts:", nonconst_tuple.__code__.co_consts)

print("\n--- 'in' literal set/frozenset optimization ---")
def check_literal(ch):
    return ch in {"a", "e", "i", "o", "u"}   # compiler isay frozenset constant bana gives is
print("co_consts:", check_literal.__code__.co_consts)
print("saw? set literal then frozenset constant bana diya -> every call but set no banta")

def check_list(ch):
    return ch in ["a", "e", "i", "o", "u"]   # list constant ban can is (tuple me)
print("list version co_consts:", check_list.__code__.co_consts)

print("\n--- string constant folding of hadd ---")
def small_str():
    return "ab" * 3             # fold be goes is
print("co_consts:", small_str.__code__.co_consts)
print("bohat big expressions (e.g. 'a'*10**9) jaan boojh kar fold no is (memory bachane of for)")

### output of matlab
`24 * 60 * 60` then compiler ne `86400` bana diya — runtime but koi multiplication no.
set literal `{"a", ...}` then `frozenset` **constant** bana diya, is for membership check
every call but nayi set no creates. this all **peephole optimizer / AST folding** of work is.

### Gotchas
- Constant folding sirf **literals** but is is — `x * 60 * 60` fold no will be.
- Micro-detail but optimize to do from first profile do; bytecode dekhna samajhne of for is.
- Bare literals of folding jaan boojh kar mehdood is (memory blow-up from bachne of for).

### Interview tip
*"CPython folds constant expressions at compile time and turns literal membership sets into
`frozenset` constants; `dis` and `co_consts` let you verify this."*

---
## 20. Local vs Global lookup speed — `LOAD_FAST` vs `LOAD_GLOBAL`

### what is
- **Local variable** function of code object of ek **array slot** me is is → `LOAD_FAST`
  (index from seedha uthao)
- **Global / builtin** then **dictionary lookup** from dhoondna parta is → `LOAD_GLOBAL`
  (module dict, phir builtins dict)

### why matter does is
Hot loop me `len()`, `append`, ya global config then baar baar dhoondhna time khaata is.
Classic trick: loop from first **local naam** me bind kar lo.

In [ ]:
import dis, timeit

GLOBAL_FACTOR = 3               # module-level global

def uses_global(n):
    total = 0
    for i in range(n):
        total += i * GLOBAL_FACTOR       # every chakkar me LOAD_GLOBAL
    return total

def uses_local(n):
    total = 0
    factor = GLOBAL_FACTOR              # ek dafa uthao
    for i in range(n):
        total += i * factor             # ab LOAD_FAST
    return total

assert uses_global(1000) == uses_local(1000)     # natija exactly same

t_g = timeit.timeit("uses_global(2000)", globals=globals(), number=300)
t_l = timeit.timeit("uses_local(2000)", globals=globals(), number=300)
print(f"global lookup: {t_g:.4f}s")
print(f"local  bind  : {t_l:.4f}s   (aam tor but thora fast)")

print("\n--- bytecode of farq ---")
print("uses_global me:", [i.opname for i in dis.get_instructions(uses_global)
                          if "GLOBAL" in i.opname or "FAST" in i.opname])
print("uses_local  me:", [i.opname for i in dis.get_instructions(uses_local)
                          if "GLOBAL" in i.opname or "FAST" in i.opname])

# method lookup then bhi bind did ja can is (classic idiom)
def append_slow(n):
    out = []
    for i in range(n):
        out.append(i)               # every dafa 'append' attribute lookup
    return out

def append_fast(n):
    out = []
    push = out.append               # bound method ek dafa uthao
    for i in range(n):
        push(i)
    return out

def append_comp(n):
    return [i for i in range(n)]    # all from fast and all from clean

assert append_slow(500) == append_fast(500) == append_comp(500)
for name, fn in [("out.append ", append_slow), ("bound push ", append_fast),
                 ("comprehension", append_comp)]:
    print(f"  {name}: {timeit.timeit(lambda fn=fn: fn(2000), number=200):.4f}s")

### output of matlab
Bytecode me clean nazar aaya: `uses_global` me `LOAD_GLOBAL` is, `uses_local` me sirf `FAST` variants.
Timing me local binding and comprehension ahead rehte are.

### Gotchas
- this **micro-optimization** is — sirf asli hot loop me do, warna code dirty is is.
- Python 3.11+ me "specializing adaptive interpreter" ne `LOAD_GLOBAL` then bohat behtar kar diya is,
  is for purani kitabon like dramatic farq no milta.
- Pehla sawal hamesha: algorithm theek is? (O(n²) → O(n)) — micro-opt after me.

### Interview tip
*"Locals are array-indexed (`LOAD_FAST`), globals are dict lookups (`LOAD_GLOBAL`). In hot loops,
binding globals/methods then locals removes repeated dictionary lookups."*

---
## 21. Common Mistakes / Gotchas — all ek jagah

Neeche every wo trap is jahan log phaste are. Cell chala kar output see.

In [ ]:
print("1) [[0]*3]*3 -> rows shared")
g = [[0] * 3] * 3; g[0][0] = 1
print("   ", g, "| sahi:", [[0] * 3 for _ in range(3)])

print("\n2) dict.fromkeys mutable default -> value shared")
d = dict.fromkeys(["a", "b"], [])       # ek hi list all keys me!
d["a"].append(1)
print("   ", d, "| sahi:", {k: [] for k in ["a", "b"]})

print("\n3) loop me list modify then do -> elements skip")
nums = [1, 2, 3, 4, 5, 6]
for x in list(nums):                    # copy but chalo
    if x % 2 == 0:
        nums.remove(x)
print("    copy but iterate (sahi):", nums)
bad = [1, 2, 3, 4, 5, 6]
for x in bad:                           # bug: index shift be goes is
    if x % 2 == 0:
        bad.remove(x)
print("    seedha iterate (bug) :", bad)

print("\n4) closure late binding -> all functions aakhri value dekhte are")
fns_bad = [lambda: i for i in range(3)]
print("    late binding:", [f() for f in fns_bad])
fns_ok = [lambda i=i: i for i in range(3)]   # default arg from value capture
print("    fixed       :", [f() for f in fns_ok])

print("\n5) tuple of inside list but += -> mutate be goes is magar Typeerror bhi aata is")
t = (["a"],)
try:
    t[0] += ["b"]                       # __iadd__ chal gaya, phir slot assign but error
except Typeerror as e:
    print("    error:", e)
print("    magar list mutate be chuki:", t)

print("\n6) is vs == numbers/strings but")
big = 1000
print("    1000 is 1000 (do naam):", big is 1000, "| == :", big == 1000)

print("\n7) sort/sorted -- ek in-place, doosra new object")
orig = [3, 1, 2]
srt = sorted(orig)                      # new object
print("    sorted():", srt, "orig:", orig)
print("    orig.sort() of return:", orig.sort(), " <- None! (in-place)")

print("\n8) mutable object then set/dict key banane of koshish")
try:
    {"key": 1}.setdefault([1], 2)
except Typeerror as e:
    print("    ", type(e).__name__, ":", e)

### every gotcha of aik-line sabaq

1. `[[0]*3]*3` → **outer repetition** references copy does is, comprehension use do.
2. `dict.fromkeys(keys, [])` → ek hi mutable value share is is; dict comprehension use do.
3. Jis list but loop chal raha is it modify no do — `list(x)` copy but chalo ya nayi list create.
4. Closure **variable** capture does is, **value** no — default arg ya `functools.partial` from bind do.
5. `t[0] += [...]` me `__iadd__` first runs is (mutation be jaati is) phir tuple assignment fail is is.
6. `is` sirf singletons of for.
7. `list.sort()` `None` gives is (in-place of convention), `sorted()` nayi list.
8. Keys hashable honi should.

---
## 22. Real-World problems

Ab in all concepts then asli masail but lagaate are — 4 problems, poore commented solutions of sath.

In [ ]:
# ---------------------------------------------------------------------------
# problem 1: 200k log lines process to do are. Team ne first list banayi
#            (memory blow-up). Generator from memory ~constant kar do.
# ---------------------------------------------------------------------------
import tracemalloc

def fake_log_lines(n):
    # asli file of jagah in-memory generator (koi outside of file no)
    for i in range(n):
        level = "error" if i % 100 == 0 else "INFO"
        yield f"2026-01-01 10:00:{i % 60:02d} {level} request id={i} took={i % 500}ms"

N = 200_000

def count_errors_list(n):
    lines = [ln for ln in fake_log_lines(n)]           # bad: all memory me
    errors = [ln for ln in lines if " error " in ln]   # bad: doosri poori list
    return len(errors)

def count_errors_stream(n):
    # ACHA: ek time me ek line -> memory O(1)
    return sum(1 for ln in fake_log_lines(n) if " error " in ln)

tracemalloc.start()
r1 = count_errors_list(N)
peak_list = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

tracemalloc.start()
r2 = count_errors_stream(N)
peak_stream = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

assert r1 == r2 == N // 100                 # natija exactly same
print("errors mile         :", r1)
print("peak memory (list)  :", round(peak_list / 1024 / 1024, 2), "MB")
print("peak memory (stream):", round(peak_stream / 1024, 1), "KB")
print("Sabaq: pipeline me generator rakho, list sirf tab jab dobara should be.")

In [ ]:
# ---------------------------------------------------------------------------
# problem 2: Default config all requests me share be rahi thi -> ek request of
#            tabdeeli all but lag gayi (classic production bug). Fix do.
# ---------------------------------------------------------------------------
import copy

DEFAULT_CONFIG = {
    "retries": 3,
    "timeouts": {"connect": 5, "read": 30},
    "features": ["cache", "gzip"],
}

def handle_request_buggy(user_overrides):
    cfg = DEFAULT_CONFIG                     # bug: same object!
    cfg.update(user_overrides)
    cfg["features"].append("debug")          # bug: shared nested list mutate
    return cfg

def handle_request_shallow(user_overrides):
    cfg = dict(DEFAULT_CONFIG)               # aadha fix: top level alag
    cfg.update(user_overrides)
    cfg["features"].append("debug")          # ABHI BHI bug: nested list shared
    return cfg

def handle_request_fixed(user_overrides):
    cfg = copy.deepcopy(DEFAULT_CONFIG)      # FIX: poora alag
    cfg.update(user_overrides)
    cfg["features"].append("debug")
    return cfg

snapshot = copy.deepcopy(DEFAULT_CONFIG)     # after me compare to do of for

handle_request_buggy({"retries": 10})
print("buggy of after global config :", DEFAULT_CONFIG["retries"], DEFAULT_CONFIG["features"])
DEFAULT_CONFIG.clear(); DEFAULT_CONFIG.update(copy.deepcopy(snapshot))   # reset

handle_request_shallow({"retries": 7})
print("shallow of after global      :", DEFAULT_CONFIG["retries"], DEFAULT_CONFIG["features"],
      " <- retries bacha, features no")
DEFAULT_CONFIG.clear(); DEFAULT_CONFIG.update(copy.deepcopy(snapshot))   # reset

cfg = handle_request_fixed({"retries": 1})
print("fixed of after global        :", DEFAULT_CONFIG["retries"], DEFAULT_CONFIG["features"])
print("request that config         :", cfg["retries"], cfg["features"])
assert DEFAULT_CONFIG == snapshot            # global exactly no chhera gaya
print("\nSabaq: shared mutable default state then every use from first deepcopy do")
print("       (ya behtar: frozen/immutable config + dataclasses.replace).")

In [ ]:
# ---------------------------------------------------------------------------
# problem 3: 50,000 rows memory me rakhni are. Naive class vs slots + interning
#            from memory kitni bachti is -- measure do.
# ---------------------------------------------------------------------------
import sys, tracemalloc

CITIES = ["Lahore", "Karachi", "Islamabad", "Peshawar", "Quetta"]
N = 50_000

class RowDict:                                  # naive: __dict__ + fresh strings
    def __init__(self, rid, city, status):
        self.rid, self.city, self.status = rid, city, status

class RowSlots:                                 # optimized: slots
    __slots__ = ("rid", "city", "status")
    def __init__(self, rid, city, status):
        self.rid, self.city, self.status = rid, city, status

def build(cls, intern_strings):
    rows = []
    for i in range(N):
        city = CITIES[i % len(CITIES)]
        status = "active" if i % 2 else "inactive"
        if intern_strings:
            city, status = sys.intern(city), sys.intern(status)
        rows.append(cls(i, city, status))
    return rows

def peak_mb(cls, intern_strings):
    tracemalloc.start()
    rows = build(cls, intern_strings)
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    n = len(rows)
    del rows
    return round(peak / 1024 / 1024, 2), n

dict_mb, n1 = peak_mb(RowDict, False)
slots_mb, n2 = peak_mb(RowSlots, True)
assert n1 == n2 == N
print(f"{N} rows:")
print("  normal class          :", dict_mb, "MB")
print("  __slots__ + interning :", slots_mb, "MB")
print(f"  bachat                : {round(100 * (1 - slots_mb / dict_mb))}%")
print("\nSabaq: bohat from chhote records be then __slots__ (ya dataclass(slots=True),")
print("       ya NamedTuple, ya columnar numpy/pandas) all from bara memory lever is.")

In [ ]:
# ---------------------------------------------------------------------------
# problem 4: Event system leak. Handlers ne subscribers then hamesha of for pakar
#            liya -> objects kabhi free no happened. weakref from fix do.
# ---------------------------------------------------------------------------
import gc, weakref

class LeakyBus:
    # bug: strong references -> subscriber kabhi no marta
    def __init__(self):
        self.handlers = []
    def subscribe(self, obj):
        self.handlers.append(obj.on_event)   # bound method -> obj then zinda holds is
    def emit(self, msg):
        return [h(msg) for h in self.handlers]

class SafeBus:
    # FIX: WeakMethod -> subscriber mar jaye then entry khud nikal do
    def __init__(self):
        self._handlers = []
    def subscribe(self, obj):
        self._handlers.append(weakref.WeakMethod(obj.on_event))
    def emit(self, msg):
        out, alive = [], []
        for ref in self._handlers:
            h = ref()                        # None = subscriber mar gaya
            if h is not None:
                out.append(h(msg)); alive.append(ref)
        self._handlers = alive               # dead entries clean
        return out
    def __len__(self):
        return len(self._handlers)

class Widget:
    def __init__(self, name): self.name = name
    def on_event(self, msg): return f"{self.name}:{msg}"

# --- leaky ---
leaky = LeakyBus()
w = Widget("w1"); leaky.subscribe(w)
watch = weakref.ref(w)
print("leaky emit:", leaky.emit("hello"))
del w; gc.collect()
print("subscriber del of after zinda?", watch() is not None, " <- LEAK")

# --- safe ---
safe = SafeBus()
w2 = Widget("w2"); safe.subscribe(w2)
watch2 = weakref.ref(w2)
print("\nsafe emit :", safe.emit("hello"), "| handlers:", len(safe))
del w2; gc.collect()
print("subscriber del of after zinda?", watch2() is not None, " <- free be gaya")
print("emit of after handlers count :", safe.emit("again"), len(safe))
print("\nSabaq: long-lived registry (bus/cache/observer) me weakref use do,")
print("       warna registry hi memory leak ban jaati is.")

---
# LeetCode problems

Is notebook of teen problems jaan boojh kar chune gaye are — teeno **references, identity and
copying** of baare me are, yaani exactly wohi jo hum ne above seekha:

| # | problem | Concept jo test is is |
|---|---|---|
| 138 | Copy list with Random Pointer | deep copy + `id()` based identity mapping |
| 133 | Clone Graph | reference cycles + visited map (deepcopy of memo khud banana) |
| 146 | LRU Cache | dict + doubly linked list (cycles!) — cache design |

### LC 138. Copy list with Random Pointer — Medium

**Link:** https://leetcode.com/problems/copy-list-with-random-pointer/

**problem (English):** Ek linked list di is jahan every node of paas `next` of ilawa ek
`random` pointer bhi is jo list of **kisi bhi** node (ya `None`) then point kar can is.
Poori list of **deep copy** create: naye nodes create, and naye nodes of `random` pointers
**naye** nodes of taraf be (purane of taraf no) — structure exactly same.

**this notebook of topic from how juda is:** this asal me `copy.deepcopy` of `memo` dict
haath from implement then do is. `old node → new node` mapping wahi "id → copy" idea is.

**Approach 1 — Hash map (2 passes)**
1. pass 1: every purane node of for new node create and `mapping[old] = new` rakho
   (`Node` hashable is default identity hash from).
2. pass 2: `mapping[old].next = mapping.get(old.next)` and
   `mapping[old].random = mapping.get(old.random)` — `.get` is for of `None` safe rahe.

- **Time:** O(n) &nbsp; **Space:** O(n) (mapping)

**Approach 2 — Interleaving (O(1) extra space)**
1. every original node of foran after uska copy ghusa do: `A → A' → B → B' → ...`
2. Ab `A'.random = A.random.next` (because kisi bhi node of copy uske foran after is) — exactly mapping of baghair!
3. Do lists then wapas alag do (original bhi restore do).

- **Time:** O(n) &nbsp; **Space:** O(1) extra (output of ilawa)

In [ ]:
class RNode:
    # LeetCode of 'Node': val + next + random
    __slots__ = ("val", "next", "random")
    def __init__(self, val, next=None, random=None):
        self.val, self.next, self.random = val, next, random

# ---- test helpers: [(val, random_index_or_None), ...] from list create / wapas pairs nikalo ----
def build_random_list(pairs):
    nodes = [RNode(v) for v, _ in pairs]
    for i, n in enumerate(nodes):
        n.next = nodes[i + 1] if i + 1 < len(nodes) else None
        ridx = pairs[i][1]
        n.random = nodes[ridx] if ridx is not None else None
    return nodes[0] if nodes else None

def to_pairs(head):
    nodes, cur = [], head
    while cur:
        nodes.append(cur); cur = cur.next
    pos = {id(n): i for i, n in enumerate(nodes)}       # identity -> index
    return [(n.val, pos.get(id(n.random)) if n.random is not None else None) for n in nodes]

def all_nodes(head):
    out, cur = [], head
    while cur:
        out.append(cur); cur = cur.next
    return out

# ---- Approach 1: hash map (deepcopy of memo haath from) ----
def copyRandomList_map(head):
    if head is None:
        return None
    mapping = {}                                # old node -> new node (yehi 'memo' is)
    cur = head
    while cur:                                  # pass 1: sirf nodes create
        mapping[cur] = RNode(cur.val)
        cur = cur.next
    cur = head
    while cur:                                  # pass 2: pointers jodo
        mapping[cur].next = mapping.get(cur.next)      # None safe
        mapping[cur].random = mapping.get(cur.random)
        cur = cur.next
    return mapping[head]

# ---- Approach 2: interleaving, O(1) extra space ----
def copyRandomList_interleave(head):
    if head is None:
        return None
    # step 1: A -> A' -> B -> B' ...
    cur = head
    while cur:
        nxt = cur.next
        cur.next = RNode(cur.val)
        cur.next.next = nxt
        cur = nxt
    # step 2: random pointers -- kisi node of copy hamesha uske foran after is
    cur = head
    while cur:
        if cur.random is not None:
            cur.next.random = cur.random.next
        cur = cur.next.next
    # step 3: alag do and original restore do
    dummy = RNode(0)
    copy_tail, cur = dummy, head
    while cur:
        copy = cur.next
        cur.next = copy.next                    # original bahaal
        copy_tail.next = copy                   # copy list barhao
        copy_tail = copy
        cur = cur.next
    return dummy.next

# ---- TESTS ----
cases = [
    [(7, None), (13, 0), (11, 4), (10, 2), (1, 0)],   # LeetCode of example 1
    [(1, 1), (2, 1)],                                  # example 2
    [(3, None), (3, 0), (3, None)],                    # example 3
    [],                                                # khaali list
    [(5, 0)],                                          # ek node jo khud then point does is
]

for pairs in cases:
    for fn in (copyRandomList_map, copyRandomList_interleave):
        head = build_random_list(pairs)
        original_nodes = all_nodes(head)
        copied = fn(head)
        # 1) structure exactly same
        assert to_pairs(copied) == pairs, (fn.__name__, pairs, to_pairs(copied))
        # 2) DEEP copy is -- koi node share no happened
        copy_ids = {id(n) for n in all_nodes(copied)}
        assert copy_ids.isdisjoint({id(n) for n in original_nodes}), "shallow copy!"
        # 3) original exactly waisa hi bacha (interleave version of for ahem)
        assert to_pairs(head) == pairs, "original kharab be gaya"

print("LC 138: both approaches of saare tests pass ✔")
print("example 1 of copy:", to_pairs(copyRandomList_map(build_random_list(cases[0]))))

**output of matlab:** Teen things sabit happened — (1) structure hu-ba-hu, (2) `id()` sets disjoint
yaani **koi node share no** (asli deep copy), (3) interleaving of after **original restore** be gaya.

**Interview follow-up:** *"Interleaving me `random` pointer without map how got?"* →
because every node of copy uske foran after is, is for `old.random.next` hi `new.random` is.
Yehi trick "invariant from memory bachana" of classic example is.

### LC 133. Clone Graph — Medium

**Link:** https://leetcode.com/problems/clone-graph/

**problem (English):** Ek connected undirected graph of node diya is (`val` + `neighbors` list).
Poore graph of **deep copy** create. Graph me **cycles** are (undirected = every edge both taraf),
is for seedha recursion infinite loop me chala jayega.

**this topic from how juda is:** exactly `deepcopy` of `memo`! Ek `visited` dict
(`old node → new node`) hi cycle protection gives is — exactly like CPython of memo.

**Approach (DFS + visited map)**
1. `visited` dict rakho: old node → new node.
2. Node but pahunchte hi **first** clone create and `visited` me daal do (recursion from **first** —
   yehi cycle-safety is, `__deepcopy__` me bhi yehi rule tha).
3. every neighbour of for recursive clone kar of naye node of `neighbors` me daalo.
4. BFS version bhi wahi work `deque` from does is (deep graph but `Recursionerror` from bachne of for behtar).

- **Time:** O(V + E) — every node and edge ek dafa
- **Space:** O(V) visited map (+ recursion stack ya BFS queue)

In [ ]:
from collections import deque

class GNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []
    def __repr__(self):
        return f"GNode({self.val})"

# ---- helpers: adjacency list (1-indexed like LeetCode) from graph create ----
def build_graph(adj):
    if not adj:
        return None
    nodes = {i + 1: GNode(i + 1) for i in range(len(adj))}
    for i, nbrs in enumerate(adj, start=1):
        nodes[i].neighbors = [nodes[j] for j in nbrs]
    return nodes[1]

def to_adj(node):
    # graph then wapas adjacency list me: {val: sorted neighbour vals}
    if node is None:
        return {}
    seen, q, out = {node.val}, deque([node]), {}
    while q:
        cur = q.popleft()
        out[cur.val] = sorted(n.val for n in cur.neighbors)
        for n in cur.neighbors:
            if n.val not in seen:
                seen.add(n.val); q.append(n)
    return out

def collect_ids(node):
    if node is None:
        return set()
    seen, q, ids = {id(node)}, deque([node]), set()
    while q:
        cur = q.popleft(); ids.add(id(cur))
        for n in cur.neighbors:
            if id(n) not in seen:
                seen.add(id(n)); q.append(n)
    return ids

# ---- Approach 1: DFS + visited (recursive) ----
def cloneGraph_dfs(node, visited=None):
    if node is None:
        return None
    visited = {} if visited is None else visited
    if node in visited:                     # first from clone be chuka -> wohi do (cycle-safe)
        return visited[node]
    clone = GNode(node.val)
    visited[node] = clone                   # LAZMI: recursion from first register do
    for nbr in node.neighbors:
        clone.neighbors.append(cloneGraph_dfs(nbr, visited))
    return clone

# ---- Approach 2: BFS (bare/deep graphs of for, koi recursion limit no) ----
def cloneGraph_bfs(node):
    if node is None:
        return None
    visited = {node: GNode(node.val)}
    q = deque([node])
    while q:
        cur = q.popleft()
        for nbr in cur.neighbors:
            if nbr not in visited:
                visited[nbr] = GNode(nbr.val)
                q.append(nbr)
            visited[cur].neighbors.append(visited[nbr])
    return visited[node]

# ---- TESTS ----
graphs = [
    [[2, 4], [1, 3], [2, 4], [1, 3]],      # LeetCode example 1 (square, cycles)
    [[]],                                   # ek akela node
    [[2], [1]],                             # do nodes
    [[2, 3], [1], [1]],                     # star
]

for adj in graphs:
    expected = to_adj(build_graph(adj))
    for fn in (cloneGraph_dfs, cloneGraph_bfs):
        original = build_graph(adj)
        clone = fn(original)
        assert to_adj(clone) == expected, (fn.__name__, adj)
        # deep copy proof: koi node share no
        assert collect_ids(clone).isdisjoint(collect_ids(original)), "shallow clone!"
        # original exactly bacha rehna should
        assert to_adj(original) == expected

assert cloneGraph_dfs(None) is None and cloneGraph_bfs(None) is None   # khaali graph
print("LC 133: DFS + BFS both pass ✔")
print("example 1 of clone adjacency:", to_adj(cloneGraph_bfs(build_graph(graphs[0]))))

# self-loop bhi handle is is (node khud of neighbour)
selfy = GNode(1); selfy.neighbors = [selfy]
c = cloneGraph_bfs(selfy)
assert c is not selfy and c.neighbors[0] is c        # clone bhi khud then point kare
print("self-loop graph bhi theek clone happened ✔")

**output of matlab:** Adjacency both me barabar, `id()` sets disjoint (poora new graph),
original mehfooz, and self-loop bhi structure of sath copy happened — matlab hamara "memo" theek work kar raha is.

**Interview follow-up:** *"`visited[node] = clone` recursion from first why?"* →
Warna cycle but wapas usi node but aa kar infinite recursion be jaayegi.
exactly yehi wajah is of `__deepcopy__` me `memo[id(self)] = new` all from first likhte are.

### LC 146. LRU Cache — Medium

**Link:** https://leetcode.com/problems/lru-cache/

**problem (English):** Ek cache create jis me `get(key)` and `put(key, value)` **O(1)** me be.
Capacity full be jaye then all from **old istemal hone that** (least recently used) item nikaal do.
`get` and `put` both item then "recently used" bana dete are.

**this topic from how juda is:** this asli production cache design is — and inside
**doubly linked list** is, yaani every node me `prev`/`next` = **reference cycles** of bharmaar.
Isi for GC (section 11) and weakref (section 13) of baat cache of sath aati is.

**Approach 1 — dict + doubly linked list (classic, "interview answer")**
- `dict`: key → node (O(1) lookup)
- DLL: ordelay (head = most recent, tail = least recent). Dummy `head`/`tail` sentinels rakho
  taake edge cases (khaali list, aakhri node) me `if` no likhna paren.
- `get`: node dhoondo → DLL from nikaal kar head of paas lagao
- `put`: maujood be then update + move; warna add; capacity cross be then `tail.prev` (LRU) hatao

**Approach 2 — `collections.OrdelayedDict`** (ya plain `dict`, jo 3.7+ me insertion-ordelayed is)
`move_to_end()` and `popitem(last=False)` from wohi work 10 lines me.

- **Time:** both me `get`/`put` **O(1)** amortized
- **Space:** O(capacity)

In [ ]:
from collections import OrdelayedDict

# ================= Approach 1: dict + doubly linked list =================
class _DNode:
    __slots__ = ("key", "val", "prev", "next")      # slots -> kam memory (section 14)
    def __init__(self, key=None, val=None):
        self.key, self.val = key, val
        self.prev = self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.map = {}                               # key -> _DNode
        # sentinels: head.next = most recent, tail.prev = least recent
        self.head, self.tail = _DNode(), _DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    # --- DLL helpers ---
    def _unlink(self, node):
        node.prev.next, node.next.prev = node.next, node.prev
        node.prev = node.next = None                # references clean (cycle no chhore)

    def _push_front(self, node):
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next = node

    # --- public API ---
    def get(self, key: int) -> int:
        node = self.map.get(key)
        if node is None:
            return -1
        self._unlink(node); self._push_front(node)   # "recently used" bana do
        return node.val

    def put(self, key: int, value: int) -> None:
        node = self.map.get(key)
        if node is not None:                         # update + move then front
            node.val = value
            self._unlink(node); self._push_front(node)
            return
        if len(self.map) >= self.cap:                # evict LRU (tail from pehla asli node)
            lru = self.tail.prev
            self._unlink(lru)
            del self.map[lru.key]
        node = _DNode(key, value)
        self.map[key] = node
        self._push_front(node)

    def keys_mru_first(self):                        # sirf testing/debug of for
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.key); cur = cur.next
        return out

# ================= Approach 2: OrdelayedDict =================
class LRUCacheOrdelayed:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.od = OrdelayedDict()

    def get(self, key: int) -> int:
        if key not in self.od:
            return -1
        self.od.move_to_end(key)                     # most-recent ban gaya
        return self.od[key]

    def put(self, key: int, value: int) -> None:
        if key in self.od:
            self.od.move_to_end(key)
        self.od[key] = value
        if len(self.od) > self.cap:
            self.od.popitem(last=False)              # all from old nikaalo

In [ ]:
# ---- TESTS (both implementations but ek like) ----
for cls in (LRUCache, LRUCacheOrdelayed):
    # LeetCode of official example
    c = cls(2)
    c.put(1, 1); c.put(2, 2)
    assert c.get(1) == 1                # 1 recently used
    c.put(3, 3)                         # capacity full -> key 2 evict
    assert c.get(2) == -1, cls.__name__
    c.put(4, 4)                         # key 1 evict (because 3,1 me 1 old)
    assert c.get(1) == -1
    assert c.get(3) == 3
    assert c.get(4) == 4

    # capacity 1 of edge case
    c1 = cls(1)
    c1.put(1, 10); assert c1.get(1) == 10
    c1.put(2, 20); assert c1.get(1) == -1 and c1.get(2) == 20

    # update from value badle magar new slot no bane
    c2 = cls(2)
    c2.put(1, 1); c2.put(1, 100); c2.put(2, 2)
    assert c2.get(1) == 100 and c2.get(2) == 2

    # get bhi recency update kare
    c3 = cls(2)
    c3.put(1, 1); c3.put(2, 2)
    c3.get(1)                           # 1 fresh be gaya
    c3.put(3, 3)                        # -> 2 evict hona should
    assert c3.get(2) == -1 and c3.get(1) == 1

print("LC 146: both implementations of saare tests pass ✔")

# ordelay of inside from muayana (sirf DLL version me available)
c = LRUCache(3)
for k in (1, 2, 3):
    c.put(k, k * 10)
c.get(1)
print("MRU -> LRU ordelay:", c.keys_mru_first(), "(1 then chhoone from wo ahead aa gaya)")

# --- interview follow-up: what functools.lru_cache kaafi no? ---
from functools import lru_cache
@lru_cache(maxsize=2)
def slow_square(n):
    return n * n
slow_square(2); slow_square(3); slow_square(2); slow_square(4)
print("\nfunctools.lru_cache stats:", slow_square.cache_info())
print("Farq: lru_cache function results cache does is (arguments = key).")
print("      LC 146 that manual cache mutable key/value store is jahan tum evict/inspect kar can be.")

**output of matlab:** both implementations exactly same behave karti are.
`keys_mru_first()` from nazar aaya of `get(1)` ne key 1 then ahead laa diya.

**Interview follow-ups:**
- *"why doubly linked list, singly no?"* → Kisi bhi node then O(1) me nikaalna is;
  singly list me uska `prev` dhoondne of for O(n) traverse then do parta.
- *"`dict` (3.7+) insertion-ordelayed is, phir DLL of need what?"* → Interview me DLL that
  jawab expected is (and `move_to_end` jaisi API plain dict me no is). Production me
  `OrdelayedDict`/`lru_cache` use do.
- *"Thread-safe how banaoge?"* → `threading.Lock` `get`/`put` of around
  (see notebook 03), ya per-shard locks.
- *"Memory leak of khatra?"* → DLL cycles creates is; `_unlink` me `prev`/`next` then `None`
  then do and bounded capacity then keep necessary is. Unbounded cache = leak.

---
# Interview questions & answers

**Q1. Python me variable what is — dabba ya label?**
Label. Names heap objects of taraf **references** are. `a = b` object copy no does,
sirf ek and naam usi object but bind does is.

**Q2. Python pass-by-value is ya pass-by-reference?**
both no — **call by object reference (sharing)**. object mutate do then caller then dikhega,
parameter rebind do then no.

**Q3. `is` and `==` me farq?**
`is` identity (same object, `id()` barabar), `==` value (`__eq__`). `is` sirf `None`/singletons/sentinels of for.

**Q4. `sys.getrefcount(x)` expected from 1 zyada why gives is?**
function call of argument khud ek temporary reference creates is.

**Q5. Immortal objects what are?**
PEP 683 (3.12+): `None`, `True`, `False`, chhote ints/strings of refcount fixed bara constant
keep diya gaya is taake unke counters but contention/locking no be — is for their refcount
ajeeb bara nazar aata is.

**Q6. CPython memory how manage does is?**
Primary: **reference counting** (refcount 0 → foran free). Sath me **cycle collector**
(generational, 3.13+ incremental) un containers of for jo cycles bana lete are.
Allocator level but `pymalloc` arenas/pools/blocks use does is and chhote objects of for free lists.

**Q7. Reference cycle what is and why masla is?**
`a.peer = b; b.peer = a` — both of refcount kabhi 0 no is, is for refcounting free no kar can.
`gc` module cycles detect kar of free does is; ya design me `weakref` use kar of cycle create hi no.

**Q8. `gc.disable()` when theek is?**
Jab tum yaqeen from jaanti be of cycles no banenge (short-lived request handlers, batch scripts)
and GC pauses measurable dard give rahe hon. Warna memory growth of risk is.

**Q9. Shallow vs deep copy?**
Shallow: new container, inside those objects shared. Deep: recursively all new, `memo` dict of sath
(cycles safe + shared structure mehfooz). Deepcopy slow is.

**Q10. `deepcopy` cycle but crash why no does?**
`memo` dict me `id(obj) → copy` holds is; dobara wohi object aane but nayi copy no creates,
memo that give gives is.

**Q11. Mutable default argument of masla and fix?**
Default `def` chalne but **ek dafa** evaluate be kar `__defaults__` me store is is —
every call me wohi object. Fix: `def f(x=None): x = [] if x is None else x`.

**Q12. `__slots__` what does is and when use do?**
Per-instance `__dict__` hata kar fixed C-array slots gives is: kam memory, thora fast attribute
access, dynamic attributes band. Sirf tab jab hazaaron/lakhon instances hon.
Subclass me `__slots__ = ()` likhna no bhoolo, and weakref should then `__weakref__` add do.

**Q13. `sys.getsizeof` list of for chhota number why gives is?**
Wo sirf container (pointer array + headelay) ginta is, inside those objects no.
Deep size of for `id()`-based `seen` set of sath recursion, ya `tracemalloc`.

**Q14. `weakref` when use do are?**
Caches/identity maps (`WeakValueDictionary`), parent back-references (cycle avoid),
observers/callbacks (`WeakMethod`), per-object metadata (`WeakKeyDictionary`).

**Q15. `__del__` why avoid then do should?**
Non-deterministic timing, exceptions nigal go are, resurrection mumkin,
shutdown but globals gaayab. Behtar: context manager (`with`) ya `weakref.finalize`.

**Q16. String concatenation loop me why slow is?**
`str` immutable — every `+=` (aam tor but) nayi string + poora copy → O(n²).
`"".join(pieces)` ek allocation me O(n).

**Q17. `list` vs `tuple` internals?**
`list`: mutable, over-allocated pointer array (amortized O(1) append), `__hash__ = None`.
`tuple`: immutable, exact size, hashable (if inside all hashable be), interpreter isay
cache/reuse kar can is, thora kam memory.

**Q18. String interning what is, `sys.intern` when?**
Compile-time identifier-jaisi strings ek hi object share karti are. `sys.intern` un cases me
faida gives is jahan bohat si duplicate runtime strings hon (log/CSV parsing) —
memory kam and dict lookups fast (pointer compare fast path). Interned strings process bhar zinda rehti are.

**Q19. `LOAD_FAST` vs `LOAD_GLOBAL`?**
Locals array-index from uthte are (fast), globals/builtins dictionary lookup from.
Hot loop me globals/methods then local naam me bind then do ek jaana-pehchana micro-optimization is.

**Q20. Constant folding what is?**
Compiler literal expressions of natija first hi nikaal takes is (`24*60*60` → `86400`),
and literal membership sets then `frozenset` constant bana gives is. `dis` / `co_consts` from see.

---
# Practice Exercises (khud do)

1. **Refcount detective** — ek function write `ref_report(obj)` jo `sys.getrefcount` and
   `gc.get_referrers` use kar of bataye kitne and who references pakre happened are.
   *Hint:* `gc.get_referrers` of output bara is is — sirf `type()` names print do.

2. **Freeze do** — `deep_freeze(obj)` write jo nested `dict`/`list` then recursively
   `MappingProxyType`/`tuple` me badal give (immutable snapshot).
   *Hint:* `types.MappingProxyType`, and recursion + `seen` set cycles of for.

3. **Copy-on-write config** — ek class create jo shared defaults but baithi be magar pehli
   write but apna deep copy bana le.
   *Hint:* `__setitem__` me flag check do, `copy.deepcopy` ek dafa.

4. **Leak hunter** — ek function jo dane but bataye of di gayi class of kitne instances zinda are.
   *Hint:* `gc.get_objects()` but `isinstance` filter, ya class me `WeakSet` registry.

5. **`__slots__` benchmark** — ek script jo `N` badal kar (1k, 10k, 100k) dict-class vs
   slots-class vs `NamedTuple` vs `dataclass(slots=True)` of memory and build time compare kare.
   *Hint:* `tracemalloc` + `time.perf_counter`, results ek table me print do.

6. **Interning of faida napo** — 100k duplicate strings of sath and without `sys.intern`
   memory measure do. Kitne percent bacha?
   *Hint:* Strings runtime but create (`"user_" + str(i % 50)`) warna compiler intern kar dega.

7. **Apna `deepcopy`** — `my_deepcopy(obj, memo=None)` write jo `dict`/`list`/`tuple`/`set`
   and custom objects handle kare, cycles safe be, and `copy.deepcopy` from compare do.
   *Hint:* first `memo[id(obj)] = new` rakho, phir bharo.

8. **Bytecode quiz** — `x in {1,2,3}` vs `x in [1,2,3]` vs `x in (1,2,3)` of `dis` compare do
   and `timeit` from speed. who jeeta and why?

9. **LC 138 variant** — same problem magar `random` pointer of sath ek `parent` pointer bhi be.
   what tumhara mapping approach change will be? (no — why? write.)

10. **LRU + TTL** — LC 146 those cache me `ttl_seconds` add do (expire be jaye then `get` but -1).
    *Hint:* node me `expires_at` rakho, `time.monotonic()` use do, lazy expiry but evict do.

---
# Summary / Cheat-sheet

## Mental model (ek line me)
> Python me **names** heap **objects** of taraf **references** are. Assignment naam bind does is,
> kabhi object copy no does. Behaviour of farq **mutability** from aata is, "value vs reference" from no.

## Quick reference

| thing | Yaad rakhne that baat |
|---|---|
| `a = b` | reference copy, object copy no |
| `a is b` | same object? (`id` barabar) — sirf singletons/sentinels of for |
| `sys.getrefcount(x)` | jawab me **+1** (argument bhi reference) |
| Immortal objects | `None`/`True`/chhote ints of refcount ajeeb bara — normal |
| Small int cache | `-5..256` pre-created |
| String interning | compile-time literals shared; runtime strings of for `sys.intern` |
| Mutation vs rebinding | `x.append()` all then dikhta; `x = ...` sirf local |
| Mutable default | `def f(a=[])` → shared state; `None` sentinel use do |
| `copy.copy` | new container, inside shared |
| `copy.deepcopy` | recursively new + `memo` (cycle safe, structure preserve) |
| `__copy__`/`__deepcopy__` | resources then copy hone from bachane of for; `memo[id(self)]` first |
| Refcounting | 0 but foran free, deterministic, cycles hal no does |
| `gc` | cycle collector, generational/incremental, sirf containers track |
| `__del__` | non-deterministic + exceptions nigal go are → `with` / `weakref.finalize` |
| `weakref` | refcount no barhata → cache/observer/parent-pointer |
| `__slots__` | `__dict__` khatam: memory kam, access fast, dynamic attrs band |
| `getsizeof` | shallow! deep size of for recursion + `seen`, ya `tracemalloc` |
| `list` | pointers + over-allocation → append amortized O(1) |
| `array`/`numpy` | raw values contiguous → kai guna kam memory |
| `deque` | both sire but O(1); indexing O(n) |
| `+=` on str in loop | O(n²) → `"".join(pieces)` ya `io.StringIO` |
| `LOAD_FAST` vs `LOAD_GLOBAL` | locals array-index, globals dict lookup |
| Constant folding | literals compile-time but hal; `in {literal set}` → `frozenset` constant |

## Memory optimization playbook (tarteeb from)
1. what data structure sahi is? (`list of dicts` → `dataclass(slots=True)` / columnar arrays)
2. all some memory me why? → **generators / streaming**
3. Duplicate strings bohat are? → `sys.intern`
4. Lakhs objects? → `__slots__` / `NamedTuple` / `array` / `numpy`
5. Cache leak kar raha is? → bounded LRU ya `WeakValueDictionary`
6. Numbers of big list? → `array`/`numpy` (pointer + boxing khatam)
7. Measure do: `tracemalloc` (where allocate happened), recursive `deep_size`, `gc.get_objects()`

## LeetCode recap
| # | Title | Core idea | Time | Space |
|---|---|---|---|---|
| 138 | Copy list with Random Pointer | old→new map (memo), ya interleaving | O(n) | O(n) / O(1) |
| 133 | Clone Graph | DFS/BFS + visited map (cycle safe) | O(V+E) | O(V) |
| 146 | LRU Cache | dict + doubly linked list (ya OrdelayedDict) | O(1) | O(capacity) |

**Agla notebook:** `02_Asyncio_Coroutines_aur_Async_Patterns.ipynb` — event loop, tasks,
and concurrency of asli patterns.